In [1]:
import math
import time

import gradio as gr
from datasets import load_dataset, Audio

from sdoml_task1.features import build_feature_dataset
from sdoml_task1.modeling.load import load_features
from sdoml_task1.modeling.train import run_training
from sdoml_task1.modeling.optimization import OPTIMIZERS, SCHEDULERS, REGULARIZATIONS, ACTIVATIONS

X_train, y_train, X_test, y_test = load_features()

Features loaded from /home/tevtev/Documents/SDOML-project1/data/processed/features.pkl


In [2]:
MAX_LAYERS = 10

def update_slider_visibility(n_layers):
    """Show only the first ``n_layers`` hidden-layer inputs."""
    updates = []
    for i in range(MAX_LAYERS):
        if i < n_layers:
            updates.append(gr.Number(visible=True))
        else:
            updates.append(gr.Number(visible=False))
    return updates


def gradio_train(epochs, lr_text, batch_size, n_layers, activation_function, regularization,
                 reg_param,  optimizer_name, scheduler_name, limit_epoch, *layer_sizes):
    """Read the values from the interface, train the model and return a summary."""
    
    try:
        lr = float(lr_text)
    except ValueError:
        raise gr.Error(f"The learning rate must be a number (e.g. 0.001, got '{lr_text}'.")
    if lr <= 0:
        raise gr.Error("The learning rate must be positive.")

    hidden_sizes = []
    for i in range(n_layers):
        if layer_sizes[i] is None or layer_sizes[i] < 1:
            raise gr.Error(f"Hidden layer {i+1} must have at least 1 neuron.")
        hidden_sizes.append(layer_sizes[i])


    if regularization == "None":
        regularization = None
        
    if regularization == "dropout":
        if reg_param < 0 or reg_param >= 1:
            raise gr.Error("For dropout, the value is between 0 and 1.")

    metrics = run_training(X_train, y_train, X_test, y_test, hidden_sizes=hidden_sizes,
                           activation_function=activation_function, regularization=regularization,
                           reg_param=reg_param, optimizer_name=optimizer_name, scheduler_name=scheduler_name,
                           epochs=int(epochs), lr=lr, batch_size=int(batch_size), limit_epoch=int(limit_epoch))

    epochs_done = len(metrics["loss_val"])
    final_acc = metrics["accuracy_val"][-1]
    best_acc = max(metrics["accuracy_val"])

    summary = (
        f"**Modele trained:**   \n"
        f"**Final validation accuracy:** {final_acc:.4f}    \n"
        f"**Best validation accuracy:** {best_acc:.4f}    \n"
        f"**Final validation loss:** {metrics['loss_val'][-1]:.4f}"
    )
    return summary

### Remember to close the port with .close() or by restarting the kernel. If you don't, it sometimes struggles to re-run the code cause the port is busy.

In [3]:
def create_app():
    with gr.Blocks() as interface:
        gr.Markdown(r"""
            # Spoken Digit Classification
            ### Project description
            This project uses the [Audio MNIST Dataset](https://huggingface.co/datasets/gilkeyio/AudioMNIST) to train a model that predicts numbers from 0 to 9 based on audio input. The neural network is a multilayer perceptron with an input of 26 neurons, two hidden layers of 64 and 32 neurons, and a final output layer of 10 neurons. It uses the Adam optimizer and the cross-entropy loss function.
        """)

        with gr.Tabs():
            # ========================================= DATA EXPLORATION TAB =========================================
            with gr.Tab("Data"):
                with gr.Group():
                    gr.Markdown("#### Dataset Exploration")

                    gr.Markdown("#### *Dummy data for testing purposes, review and update*")

                    gr.Markdown("""
                    The Audio MNIST dataset contains 3,000 audio recordings of spoken digits (0-9) from 60 different speakers. 
                    Each recording is approximately 1 second long and sampled at 8 kHz. The dataset is split into training (2,400 samples) 
                    and testing (600 samples) sets with a balanced distribution across all digit classes.
                    """)
                    
                    gr.Markdown("**Dataset Statistics:**")
                    
                    # Dummy table data
                    table_data = [
                        ["Total Samples", "3000"],
                        ["Training Samples", "2400"],
                        ["Testing Samples", "600"],
                        ["Number of Classes", "10 (digits 0-9)"],
                        ["Sample Rate", "8 kHz"],
                        ["Average Duration", "1.0 seconds"],
                        ["Speakers", "60"],
                    ]
                    
                    gr.Dataframe(
                        value=table_data,
                        headers=["Metric", "Value"],
                        interactive=False,
                        label="Dataset Overview"
                    )
                    

            # ======================================== TRAINING INTERFACE TAB ========================================

            with gr.Tab("Train"):
                gr.Markdown("#### Training Interface")

                gr.Markdown("Choose the settings, then click **Train**. Note that if the number of epoch is high, it may take lot of time")
                with gr.Row():
                    with gr.Column(scale=2):
                        with gr.Row():
                            epochs_input = gr.Number(value=10, minimum=1, precision=0, label="Epochs")
                            lr_input = gr.Textbox(value="0.001", label="Learning rate")
                            batch_input = gr.Number(value=64, minimum=1, precision=0, label="Batch size")

                        n_layers_slider = gr.Slider(minimum=1, maximum=MAX_LAYERS, value=1, step=1, label="Number of hidden layers")
                        
                        with gr.Row():
                            layer_inputs = []
                            for i in range(MAX_LAYERS):
                                layer_input = gr.Number(value=64, minimum=1, precision=0, label=f"Layer {i+1} neurons", visible=(i == 0))
                                layer_inputs.append(layer_input)

                        activation = gr.Radio(ACTIVATIONS, value="relu", label="Activation function")
                        optimizer = gr.Radio(OPTIMIZERS, value="Adam", label="Optimizer")
                        scheduler = gr.Radio(SCHEDULERS, value="None", label="Learning rate scheduler")
                        regularization = gr.Radio(REGULARIZATIONS, value="None", label="Regularization")
                        
                        with gr.Row():
                            reg_param_input = gr.Number(value=0.0, minimum=0, label="Regularization strength", info="L1 / weight decay / dropout probability")
                            patience_input = gr.Number(value=10, minimum=1, precision=0, label="Early stopping patience", info="Only early stopping")

                        train_button = gr.Button("Train", variant="primary")

                    
                    with gr.Column(scale=1):
                        summary_output = gr.Markdown("*Results will appear here after training.*")

               
                n_layers_slider.change(fn=update_slider_visibility, inputs=n_layers_slider, outputs=layer_inputs)

                train_inputs = [epochs_input, lr_input, batch_input, n_layers_slider, activation, regularization, reg_param_input,
                                optimizer, scheduler, patience_input, *layer_inputs]
                train_button.click(fn=gradio_train, inputs=train_inputs, outputs=summary_output)



            # ========================================= MODEL EVALUATION TAB =========================================
            with gr.Tab("Model"):
                gr.Markdown("#### Model Evaluation")

        gr.Markdown(r"""
            *Created by: Amets Martiarena, Teva Philippe, Alvaro Crespo | Check the original repo [here](https://github.com/AmetsMarti/SDOML-project1)*.
        """)
    return interface

In [4]:
try:
    interface.close()
except:
    pass

# Set to true to publish the app
send_it = False

interface = create_app()
interface.launch(server_port=7860, share=send_it)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
